# Backbone+RL: Run Evaluation & Analyze Algebraic Connectivity

This notebook:
1. Loads trained policy weights from `local-checkpoints/checkpoints/`
2. Runs the backbone+RL evaluation over a grid of graph sizes $n$ and densities $\rho$
3. Saves results to CSV
4. Generates publication-quality figures of algebraic connectivity ($\lambda_2$)

> **Paper**: "Graph Theory Guided Reinforcement Learning for Network Design with High Algebraic Connectivity under Vertex and Edge Constraints" (CDC 2026)

## 1. Setup: Install Dependencies & Import Libraries

Ensure all required packages are installed, then import them.

In [ ]:
import sys, os, subprocess, json, time
from pathlib import Path

# If running in Colab, mount Drive and set project root
if "google.colab" in str(get_ipython()):
    from google.colab import drive
    drive.mount("/content/drive")
    # Adjust this to where you uploaded the project
    PROJECT_ROOT = Path("/content/drive/MyDrive/Algebraic-Connectivity-Optimization-CDC2026")
else:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

# Install dependencies if needed
# !pip install -r requirements-rl.txt 2>&1 | tail -5

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline

plt.rcParams.update({
    "figure.dpi": 150, "font.size": 11, "axes.titlesize": 13,
    "axes.labelsize": 12, "legend.fontsize": 10, "lines.linewidth": 1.5,
})
print("All imports ready.")

## 2. Configure Experiment Parameters

Set the graph sizes, density grid, number of repeats, checkpoint path, and output directory.

In [ ]:
# ── Paths ──
CHECKPOINT = PROJECT_ROOT / ".." / "local-checkpoints" / "checkpoints" / "best.pt"
CONFIG = PROJECT_ROOT / "configs" / "rl_cpu_standard.yaml"
OUT_DIR = PROJECT_ROOT / "results" / "rl"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR = PROJECT_ROOT / "figures"
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Grid ──
# Full paper grid: n_values = [8, 16, 32, 64, 128], points_per_n=100, repeats=5
# For quick testing, use a smaller grid:
QUICK_TEST = False  # Set to False for full paper experiment

if QUICK_TEST:
    N_VALUES = [8, 16, 32]
    POINTS_PER_N = 10
    REPEATS = 2
    SEED_START = 0
else:
    N_VALUES = [8, 16, 32, 64, 128]
    POINTS_PER_N = 100
    REPEATS = 5
    SEED_START = 0

SEEDS = list(range(SEED_START, SEED_START + REPEATS))
DEVICE = "cpu"

print(f"Grid: n={N_VALUES}")
print(f"Points per n: {POINTS_PER_N}, Repeats: {REPEATS}")
print(f"Checkpoint: {CHECKPOINT}")
print(f"Checkpoint exists: {CHECKPOINT.exists()}")
print(f"Config exists: {CONFIG.exists()}")

# ── Backbone args (from paper) ──
BACKBONE_ARGS = {
    "cayley_index2_overlap_high": 0.6,
    "cayley_index3_overlap_high": 2.0 / 3.0,
    "cayley_index4_overlap_low": 2.0 / 3.0,
    "cayley_multi_index_overlap": True,
    "cayley_enable_index4": True,
}

## 3. Run Path+RL and Backbone+RL Evaluations

This calls `rl_train.run_path_vs_backbone_experiment` which internally runs `rl_train.evaluate` for each $(n, \rho)$ combination with both path and backbone initialization modes. Results are saved to CSV.

The experiment:
- **Path+RL**: starts from a simple path graph, then RL adds edges greedily
- **Backbone+RL**: starts from a density-selected backbone (Cayley or Envelope), then RL adds edges greedily

The RL policy uses the same trained checkpoint (`best.pt`) for both modes — only the initialization differs.

In [ ]:
# ── Check paths before running ──
CHECKPOINT = CHECKPOINT.resolve()
print(f"Resolved checkpoint: {CHECKPOINT}")
print(f"Checkpoint exists: {CHECKPOINT.exists()}")
print(f"Config exists: {CONFIG.exists()}")

if not CHECKPOINT.exists():
    # Search for checkpoint files in parent directories
    found = list(PROJECT_ROOT.parent.rglob("best.pt"))
    if not found:
        found = list(PROJECT_ROOT.parent.rglob("*.pt"))
    if found:
        print(f"Found checkpoint(s): {found}")
        CHECKPOINT = found[0]
    else:
        print(f"No .pt files found under {PROJECT_ROOT.parent}")
        # Try the project itself
        found = list(PROJECT_ROOT.rglob("*.pt"))
        if found:
            print(f"Found checkpoint(s) in project: {found}")
            CHECKPOINT = found[0]
        else:
            raise FileNotFoundError(f"No checkpoint found. Upload checkpoints to: {CHECKPOINT}")

print(f"\nUsing checkpoint: {CHECKPOINT}")

# ── Test that the evaluate module imports correctly ──
import importlib, traceback
try:
    import rl_train.evaluate as eval_mod
    importlib.reload(eval_mod)
    print("✓ rl_train.evaluate imported successfully")
except Exception as e:
    print(f"✗ Import failed: {e}")
    traceback.print_exc()

# ── Test with a single small evaluate call (diagnostic) ──
print("\n" + "=" * 72)
print("DIAGNOSTIC: Running single evaluate call for n=8, path mode")
print("=" * 72)

diag_cmd = [
    sys.executable, "-m", "rl_train.evaluate",
    "--config", str(CONFIG),
    "--checkpoint", f"best={CHECKPOINT}",
    "--n-values", "8",
    "--densities", "0.0476,0.0952",
    "--seeds", "0",
    "--device", DEVICE,
    "--progress-every", "1",
    "--init-mode", "path",
    "--out", str(OUT_DIR / "diag.csv"),
]
print(" ".join(diag_cmd))
diag = subprocess.run(diag_cmd, capture_output=True, text=True, cwd=PROJECT_ROOT)
print("\nSTDOUT:", diag.stdout[-2000:])
print("\nSTDERR:", diag.stderr[-2000:])
print(f"Return code: {diag.returncode}")

## 3b. Run Full Experiment (Path+RL and Backbone+RL)

The diagnostic above passed, so now run the **full grid**: $n \in \{8, 16, 32, 64, 128\}$, 100 density points per $n$, 5 seeds each, for **both** Path+RL and Backbone+RL modes.

**Expected runtime**: ~1–2 hours on Colab CPU (n=128 episodes are the bottleneck).

In [ ]:
import concurrent.futures, csv
from rl_train.run_path_vs_backbone_experiment import _nontrivial_m_points, _compute_cost_summary, _compute_quality_summary

CHECKPOINT = CHECKPOINT.resolve()

print("=" * 72)
print("RUNNING BACKBONE+RL EXPERIMENT (PARALLELIZED)")
print(f"Grid: n={N_VALUES}, points_per_n={POINTS_PER_N}, repeats={REPEATS}")
print(f"Checkpoint: {CHECKPOINT}")
total_jobs = sum(len(_nontrivial_m_points(n, POINTS_PER_N)) * REPEATS * 2 for n in N_VALUES)
print(f"Total episodes: {total_jobs}")
print("=" * 72)

def run_one_eval(n, init_mode):
    """Run evaluate.py for one (n, init_mode). Returns (n, init_mode, success, csv_path)."""
    m_points = _nontrivial_m_points(n, POINTS_PER_N)
    rho_values = [(m - (n - 1)) / ((n * (n - 1)) // 2 - (n - 1)) for m in m_points]
    out_csv = OUT_DIR / "per_n_mode" / f"eval_{init_mode}_n{n}.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, "-m", "rl_train.evaluate",
        "--config", str(CONFIG),
        "--checkpoint", f"best={CHECKPOINT}",
        "--n-values", str(n),
        "--densities", ",".join(f"{r:.12f}" for r in rho_values),
        "--seeds", ",".join(str(s) for s in SEEDS),
        "--device", DEVICE,
        "--progress-every", "9999",
        "--quiet",
        "--init-mode", init_mode,
        "--out", str(out_csv),
    ]
    if init_mode == "backbone":
        cmd += (["--cayley-multi-index-overlap"] if BACKBONE_ARGS["cayley_multi_index_overlap"]
                else ["--no-cayley-multi-index-overlap"])
        cmd += (["--cayley-enable-index4"] if BACKBONE_ARGS["cayley_enable_index4"]
                else ["--no-cayley-enable-index4"])
        cmd += ["--cayley-index2-overlap-high", str(BACKBONE_ARGS["cayley_index2_overlap_high"])]
        cmd += ["--cayley-index3-overlap-high", str(BACKBONE_ARGS["cayley_index3_overlap_high"])]
        cmd += ["--cayley-index4-overlap-low", str(BACKBONE_ARGS["cayley_index4_overlap_low"])]

    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT)
    elapsed = time.time() - t0
    ok = result.returncode == 0
    print(f"  [{'OK' if ok else 'FAIL'}] n={n}, {init_mode}, {elapsed:.1f}s"
          + (f"  (stderr: {result.stderr[-200:]})" if not ok and result.stderr else ""))
    return n, init_mode, ok, str(out_csv)

# Build jobs: each (n, init_mode) pair runs in parallel
jobs = [(n, mode) for n in N_VALUES for mode in ("path", "backbone")]
max_workers = min(len(jobs), os.cpu_count() or 4)
print(f"\nRunning {len(jobs)} jobs with {max_workers} workers...")
t_start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
    futures = {pool.submit(run_one_eval, n, mode): (n, mode) for n, mode in jobs}
    results = {}
    for future in concurrent.futures.as_completed(futures):
        n, mode, ok, out = future.result()
        results[(n, mode)] = (ok, out)

elapsed = time.time() - t_start

# Merge results
raw_rows = []
first_fields = None
for n in N_VALUES:
    for mode in ("path", "backbone"):
        ok, out_csv = results.get((n, mode), (False, ""))
        if not ok or not Path(out_csv).exists():
            print(f"  ⚠ Missing: n={n}, mode={mode}")
            continue
        with open(out_csv, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                enriched = dict(row)
                enriched["method"] = "RL+Backbone" if mode == "backbone" else "RL+Path"
                enriched["repeat_idx"] = int(float(row["seed"]))
                raw_rows.append(enriched)
                if first_fields is None:
                    first_fields = list(row.keys())

OUT_DIR.mkdir(parents=True, exist_ok=True)
raw_fields = list(first_fields or [])
for col in ("method", "repeat_idx"):
    if col not in raw_fields:
        raw_fields.append(col)
with open(OUT_DIR / "raw.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=raw_fields)
    w.writeheader(); w.writerows(raw_rows)
print(f"\nCombined {len(raw_rows)} rows → {OUT_DIR / 'raw.csv'}")

# Summary CSVs
cost_rows = _compute_cost_summary(raw_rows)
qual_rows = _compute_quality_summary(raw_rows)
with open(OUT_DIR / "summary_cost.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(cost_rows[0].keys()))
    w.writeheader(); w.writerows(cost_rows)
with open(OUT_DIR / "summary_quality.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(qual_rows[0].keys()))
    w.writeheader(); w.writerows(qual_rows)

with open(OUT_DIR / "raw.meta.json", "w") as f:
    json.dump({"checkpoint": str(CHECKPOINT), "n_values": N_VALUES,
               "points_per_n": POINTS_PER_N, "repeats": REPEATS,
               "total_rows": len(raw_rows), "elapsed_sec": round(elapsed, 1)}, f, indent=2)
print(f"Done. Wall time: {elapsed:.1f}s (vs ~15min sequential)")

## 4. Load Results & Prepare for Plotting

Load the freshly-generated CSV, inspect its structure, and define helper functions for consistent plotting.

In [ ]:
# Try loading results from the new experiment run first
raw_path = OUT_DIR / "raw.csv"
summary_q_path = OUT_DIR / "summary_quality.csv"
summary_c_path = OUT_DIR / "summary_cost.csv"

if not raw_path.exists():
    # Fallback to previous results
    prev = PROJECT_ROOT / "results_prev" / "rl"
    if (prev / "raw.csv").exists():
        print(f"New results not found at {raw_path}")
        print(f"Falling back to previous results at {prev}")
        OUT_DIR = prev
        raw_path = prev / "raw.csv"
        summary_q_path = prev / "summary_quality.csv"
        summary_c_path = prev / "summary_cost.csv"
    else:
        raise FileNotFoundError(
            f"No results found. Run Cell 3b (Full Experiment) first, "
            f"or ensure results exist at {raw_path} or {prev / 'raw.csv'}"
        )

raw = pd.read_csv(raw_path)

if summary_q_path.exists() and summary_c_path.exists():
    summary_q = pd.read_csv(summary_q_path)
    summary_c = pd.read_csv(summary_c_path)
    print("Summary files loaded.")
else:
    print("Note: summary files not found — will be generated by the experiment.")
    summary_q = summary_c = None

print(f"Raw data: {len(raw)} rows, {len(raw.columns)} cols")
print(f"Methods: {raw['method'].unique().tolist()}")
print(f"n values: {sorted(raw['n'].unique())}")
raw.head(3)

# Split
bb = raw[raw["method"] == "RL+Backbone"].copy()
path_only = raw[raw["method"] == "RL+Path"].copy()
print(f"\nRL+Backbone: {len(bb)} rows")
print(f"RL+Path:     {len(path_only)} rows")

# Consistent styling
N_ORDER = sorted(raw["n"].unique())
COLORS = {8: "#4C72B0", 16: "#DD8452", 32: "#55A868", 64: "#C44E52", 128: "#8172B2"}
MARKERS = {8: "o", 16: "s", 32: "D", 64: "^", 128: "v"}

def group_agg(df, xcol, ycol, group_cols):
    g = df.groupby(group_cols).agg(
        mean=(ycol, "mean"), std=(ycol, "std"), count=(ycol, "count")
    ).reset_index()
    return g

## 5. Figure 1: $\lambda_2$ vs Target Density (per $n$, BB+RL)

Algebraic connectivity $\lambda_2$ as a function of target density $\rho$, with one subplot per graph size $n$. Error bands show $\pm 1$ std across repeats.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes_flat = axes.flatten()
for idx, n in enumerate(N_ORDER):
    ax = axes_flat[idx]
    sub = bb[bb["n"] == n]
    grp = group_agg(sub, "rho_target", "terminal_lambda2", ["rho_target"])
    ax.errorbar(grp["rho_target"], grp["mean"], yerr=grp["std"],
                fmt=f"-{MARKERS[n]}", color=COLORS[n], capsize=3, markersize=5,
                label=f"n={n}")
    ax.set_xlabel(r"Target density $\rho$")
    ax.set_ylabel(r"$\lambda_2$ (algebraic connectivity)")
    ax.set_title(f"n = {n}")
    ax.legend()
    ax.grid(True, alpha=0.3)
axes_flat[-1].set_visible(False)
fig.suptitle(r"Backbone+RL: $\lambda_2$ vs Target Density", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig1_lambda2_vs_rho.png", bbox_inches="tight")
plt.show()
print("Figure 1 saved.")

## 6. Figure 2: Normalized $\lambda_2 / n$ vs Density (BB+RL)

All $n$ overlaid on one plot, showing the normalized algebraic connectivity.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for n in N_ORDER:
    sub = bb[bb["n"] == n]
    grp = group_agg(sub, "rho_target", "terminal_lambda2_norm", ["rho_target"])
    ax.errorbar(grp["rho_target"], grp["mean"], yerr=grp["std"],
                fmt=f"-{MARKERS[n]}", color=COLORS[n], capsize=3, markersize=5,
                label=f"n={n}")
ax.set_xlabel(r"Target density $\rho$")
ax.set_ylabel(r"$\lambda_2 / n$ (normalized)")
ax.set_title(r"Backbone+RL: Normalized $\lambda_2$ vs Density")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig2_lambda2norm_vs_rho.png", bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

## 7. Figure 3: $\lambda_2$ Improvement over Initial Backbone

How much does the RL agent improve $\lambda_2$ beyond what the backbone provides? Left: absolute $\Delta\lambda_2 = \lambda_2^{\text{final}} - \lambda_2^{\text{init}}$. Right: percentage improvement.

In [ ]:
bb_plot = bb.copy()
bb_plot["lambda2_improvement"] = bb_plot["terminal_lambda2"] - bb_plot["init_lambda2"]
bb_plot["lambda2_improvement_pct"] = (
    (bb_plot["terminal_lambda2"] - bb_plot["init_lambda2"]) / bb_plot["init_lambda2"].clip(lower=1e-10) * 100
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax_idx, (col, ylabel, title) in enumerate([
    ("lambda2_improvement", r"$\Delta\lambda_2$", "Absolute $\lambda_2$ Improvement"),
    ("lambda2_improvement_pct", r"$\Delta\lambda_2 / \lambda_2^{\text{init}}$ (%)", "Relative $\lambda_2$ Improvement"),
]):
    ax = axes[ax_idx]
    for n in N_ORDER:
        sub = bb_plot[bb_plot["n"] == n]
        grp = group_agg(sub, "rho_target", col, ["rho_target"])
        ax.errorbar(grp["rho_target"], grp["mean"], yerr=grp["std"],
                    fmt=f"-{MARKERS[n]}", color=COLORS[n], capsize=3, markersize=5, label=f"n={n}")
    ax.set_xlabel(r"Target density $\rho$")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
fig.suptitle(r"Backbone+RL: $\lambda_2$ Improvement over Initial Backbone", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig3_lambda2_improvement.png", bbox_inches="tight")
plt.show()
print("Figure 3 saved.")

## 8. Figure 4: BB+RL vs Path+RL Comparison

Side-by-side comparison showing the advantage of backbone initialization over a simple path graph initialization.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()
for idx, n in enumerate(N_ORDER):
    ax = axes_flat[idx]
    for label, grp_df, color in [
        ("RL+Path", path_only, "#888888"),
        ("RL+Backbone", bb, COLORS[n]),
    ]:
        sub = grp_df[grp_df["n"] == n]
        grp = group_agg(sub, "rho_target", "terminal_lambda2", ["rho_target"])
        ax.plot(grp["rho_target"], grp["mean"],
                marker=MARKERS[n] if label == "RL+Backbone" else ".",
                color=color, label=label, alpha=0.85)
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$\lambda_2$")
    ax.set_title(f"n={n}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes_flat[-1].set_visible(False)
fig.suptitle(r"$\lambda_2$ Comparison: Backbone+RL vs Path+RL", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig4_bb_vs_path_lambda2.png", bbox_inches="tight")
plt.show()
print("Figure 4 saved.")

## 9. Figure 5: Heatmap of $\lambda_2$ across $(n, \rho)$ — BB+RL

A compact overview of mean $\lambda_2$ across the full experimental grid. Density is binned into 20 groups for readable axes.

In [ ]:
bb_plot = bb.copy()
bb_plot["rho_bin"] = pd.cut(bb_plot["rho_target"], bins=20, labels=False) / 20
bin_edges = pd.cut(bb["rho_target"], bins=20).cat.categories
bin_centers = [(interval.left + interval.right) / 2 for interval in bin_edges]
bb_pivot = bb_plot.pivot_table(
    index="n", columns="rho_bin", values="terminal_lambda2", aggfunc="mean"
)
bb_pivot = bb_pivot.reindex(N_ORDER)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(bb_pivot.values, aspect="auto", cmap="viridis", origin="lower", interpolation="nearest")
tick_step = max(1, len(bin_centers) // 10)
ax.set_xticks(range(0, len(bin_centers), tick_step))
ax.set_xticklabels([f"{bin_centers[i]:.2f}" for i in range(0, len(bin_centers), tick_step)])
ax.set_yticks(range(len(N_ORDER)))
ax.set_yticklabels([f"n={int(n)}" for n in N_ORDER])
ax.set_xlabel(r"Target density $\rho$")
ax.set_title(r"Mean $\lambda_2$ across $(n, \rho)$ — Backbone+RL")
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label(r"$\lambda_2$")
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig5_heatmap_lambda2.png", bbox_inches="tight")
plt.show()
print("Figure 5 saved.")

## 10. Figure 6: Runtime Scaling with Graph Size

Total runtime per episode (backbone construction + RL edge additions) as a function of $n$. Error bars show variation across density levels.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for _, row in summary_c.iterrows():
    n = int(row["n"])
    meth = row["method"]
    if meth == "RL+Path":
        continue
    ax.errorbar(n, row["runtime_mean_over_density_s"],
                yerr=row["runtime_std_over_density_s"],
                fmt=MARKERS.get(n, "o"), color=COLORS.get(n, "#333"),
                capsize=3, markersize=8, label=f"n={n}")
ax.set_xlabel("Graph size $n$")
ax.set_ylabel("Runtime (s)")
ax.set_title("Backbone+RL: Runtime per Episode vs $n$")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(N_ORDER)
ax.set_xticklabels([str(v) for v in N_ORDER])
ax.legend()
ax.grid(True, alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig6_runtime.png", bbox_inches="tight")
plt.show()
print("Figure 6 saved.")

## 11. Figure 7: $\lambda_2$ by Backbone Family (Cayley vs Envelope)

Splitting Backbone+RL results by the backbone family used. The routing policy selects which family (Cayley or Envelope) to use based on density $\rho$.

In [ ]:
FAMILY_COLORS = {"cayley": "#E24A33", "envelope": "#348ABD"}
FAMILY_MARKERS = {"cayley": "^", "envelope": "s"}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()
for idx, n in enumerate(N_ORDER):
    ax = axes_flat[idx]
    sub = bb[bb["n"] == n]
    for fam in ["cayley", "envelope"]:
        fam_df = sub[sub["init_family"] == fam]
        if fam_df.empty:
            continue
        grp = group_agg(fam_df, "rho_target", "terminal_lambda2", ["rho_target"])
        ax.errorbar(grp["rho_target"], grp["mean"], yerr=grp["std"],
                    fmt=f"-{FAMILY_MARKERS[fam]}", color=FAMILY_COLORS[fam],
                    capsize=3, markersize=5, label=f"{fam}")
    ax.set_xlabel(r"Target density $\rho$")
    ax.set_ylabel(r"$\lambda_2$")
    ax.set_title(f"n = {n}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes_flat[-1].set_visible(False)
fig.suptitle(r"BB+RL: $\lambda_2$ by Backbone Family (Cayley vs Envelope)", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig7_lambda2_by_family.png", bbox_inches="tight")
plt.show()
print("Figure 7 saved.")

## 12. Figure 8: Summary Panel

6-panel overview aggregating key metrics for Backbone+RL across all $(n,\rho)$.

In [ ]:
fig = plt.figure(figsize=(16, 12))
def plot_metric(ax, ycol, ylabel, title):
    for n in N_ORDER:
        sub = bb[bb["n"] == n].copy()
        if ycol not in sub.columns and ycol == "lambda2_improvement_pct":
            sub["lambda2_improvement_pct"] = (sub["terminal_lambda2"] - sub["init_lambda2"]) / sub["init_lambda2"].clip(lower=1e-10) * 100
            ycol_use = "lambda2_improvement_pct"
        else:
            ycol_use = ycol
        grp = group_agg(sub, "rho_target", ycol_use, ["rho_target"])
        ax.plot(grp["rho_target"], grp["mean"],
                marker=MARKERS[n], color=COLORS[n], label=f"n={n}")
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

panels = [
    (1, "terminal_lambda2", r"$\lambda_2$", r"$\lambda_2$ vs $\rho$"),
    (2, "terminal_lambda2_norm", r"$\lambda_2 / n$", r"Normalized $\lambda_2$"),
    (3, "episode_len", "Edges added by RL", "Episode Length"),
    (4, "init_lambda2", r"$\lambda_2^{\text{init}}$", "Initial Backbone $\lambda_2$"),
    (5, "lambda2_improvement_pct", r"$\Delta\lambda_2 / \lambda_2^{\text{init}}$ (%)", "Relative Improvement"),
    (6, "rl_runtime_sec", "Runtime (s)", "RL Runtime"),
]
for pos, col, ylbl, title in panels:
    ax = fig.add_subplot(2, 3, pos)
    plot_metric(ax, col, ylbl, title)
fig.suptitle("Backbone+RL Performance Summary", fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig(FIGS_DIR / "fig8_summary_panel.png", bbox_inches="tight")
plt.show()
print("Figure 8 saved.")

## 13. Key Quantitative Findings

Summary statistics comparing Backbone+RL against Path+RL from the quality summary table.

In [ ]:
print("=" * 72)
print("KEY QUANTITATIVE FINDINGS — Backbone+RL vs Path+RL")
print("=" * 72)

bb_summary = summary_q[summary_q["method"] == "RL+Backbone"]
print(f"\n{'n':>4}  {'Δλ₂ mean':>12}  {'Δλ₂ % mean':>12}  {'Win% vs Path':>14}  {'Points':>8}")
print("-" * 56)
for _, row in bb_summary.iterrows():
    n = int(row["n"])
    delta_mean = row["delta_lambda2_mean_over_density"]
    delta_pct = row["delta_lambda2_pct_mean_over_density"]
    win_pct = row["win_pct_vs_rl_path_over_all_points"]
    pts = int(row["total_points"])
    print(f"{n:>4}  {delta_mean:>12.4f}  {delta_pct:>12.2f}%  {win_pct:>13.1f}%  {pts:>8}")

print(f"\nTotal evaluation rows: {len(raw)}")
print(f"Figures saved to: {FIGS_DIR}/")
print("Done.")